# Johansen CO2 Storage — Bayesian Optimization Pipeline

**Objective:** Find optimal injection rate `Q` and time window `(T_start, T_end)` for each active injector well to **maximize total CO2 stored** in the Johansen formation, subject to:
- Peak BHP <= 248 bar (approx 360 psi) — seal fracture constraint
- Physical plausibility: T_start < T_end, Q > 0

**Method:** Bayesian Optimization via `Optuna` (TPE surrogate) calling MATLAB MRST simulations.

**Active Wells Optimized:** `31/01/01`, `31/1-3 S`, `31/2-5`, `31/05/02`, `31/05/07`

> Each trial runs a full MRST simulation (~5-10 min). Plan for overnight execution with N_TRIALS=40.

In [1]:
# Cell 1: Install dependencies
import subprocess, sys

pkgs = ['optuna', 'scikit-learn']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-','_'))
        print(f'  ok {pkg} already installed')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  ok {pkg} installed')

print('All dependencies ready.')

  Installing optuna...



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


  ok optuna installed
  Installing scikit-learn...
  ok scikit-learn installed
All dependencies ready.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Cell 2: Imports and Configuration
import os, re, glob, shutil, time, json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings; warnings.filterwarnings('ignore')

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# PATHS
MATLAB_BIN     = '/Applications/MATLAB_R2025b.app/bin/matlab'
MRST_ROOT      = '/Users/apple/Desktop/study/programming/Matlab/Plugins/MRST-2026a'
SCRIPT_PATH    = f'{MRST_ROOT}/co2lab/co2lab-ve/examples/example3DJohansen.m'
WELL_PLAN_PATH = f'{MRST_ROOT}/core/examples/data/Johansen/data/well_plan.csv'
WELL_CSVS_ROOT = f'{MRST_ROOT}/core/examples/data/Johansen/well_csvs'
RESULTS_JSON   = f'{MRST_ROOT}/core/examples/data/Johansen/python/bo_results.json'
BACKUP_CSV     = f'{MRST_ROOT}/core/examples/data/Johansen/data/well_plan_BACKUP_BO.csv'

# OPTIMIZATION CONFIG
N_TRIALS       = 40       # Total trials
N_WARMUP       = 10       # Random warm-up before BO-guided
MATLAB_TIMEOUT = 1200     # Max seconds per MATLAB run (20 min)
PENALTY_LAMBDA = 100.0    # Penalty per bar over BHP limit
BHP_LIMIT_BAR  = 248.0    # 360 psi in bar

# ACTIVE WELLS
DEEP_WELLS    = ['31/01/01', '31/1-3 S', '31/2-5', '31/05/02']
CENTRAL_WELL  = '31/05/07'

# SEARCH SPACE BOUNDS
BOUNDS = {
    'Q_deep':       (0.3, 3.0),
    'Q_central':    (0.3, 3.0),
    'T_start_deep': (0,   20),
    'T_end_deep':   (30,  300),
    'T_start_cen':  (0,   20),
    'T_end_cen':    (30,  300),
}

print('Configuration loaded.')
print(f'  MATLAB: {MATLAB_BIN}')
print(f'  N_TRIALS: {N_TRIALS} ({N_WARMUP} random + {N_TRIALS-N_WARMUP} BO-guided)')
print(f'  BHP limit: {BHP_LIMIT_BAR} bar ({BHP_LIMIT_BAR*14.5038:.0f} psi)')

Configuration loaded.
  MATLAB: /Applications/MATLAB_R2025b.app/bin/matlab
  N_TRIALS: 40 (10 random + 30 BO-guided)
  BHP limit: 248.0 bar (3597 psi)


In [3]:
# Cell 3: Utility Functions

def backup_well_plan():
    if not os.path.exists(BACKUP_CSV):
        shutil.copy2(WELL_PLAN_PATH, BACKUP_CSV)
        print(f'Backup saved -> {BACKUP_CSV}')
    else:
        print(f'Backup already exists -> {BACKUP_CSV}')

def restore_well_plan():
    if os.path.exists(BACKUP_CSV):
        shutil.copy2(BACKUP_CSV, WELL_PLAN_PATH)
        print('Original well_plan.csv restored from backup.')
    else:
        print('No backup found.')

def write_well_plan(Q_deep, T_start_deep, T_end_deep, Q_central, T_start_cen, T_end_cen):
    df = pd.read_csv(BACKUP_CSV)
    T_start_deep = int(T_start_deep)
    T_end_deep   = max(int(T_end_deep), T_start_deep + 10)
    T_start_cen  = int(T_start_cen)
    T_end_cen    = max(int(T_end_cen), T_start_cen + 10)
    for well in DEEP_WELLS:
        mask = df['Well_Bore_Name'] == well
        if mask.any():
            df.loc[mask, 'Rate_MtPerYear'] = round(Q_deep, 4)
            df.loc[mask, 'Start_Year']     = T_start_deep
            df.loc[mask, 'End_Year']       = T_end_deep
    mask_cen = df['Well_Bore_Name'] == CENTRAL_WELL
    if mask_cen.any():
        df.loc[mask_cen, 'Rate_MtPerYear'] = round(Q_central, 4)
        df.loc[mask_cen, 'Start_Year']     = T_start_cen
        df.loc[mask_cen, 'End_Year']       = T_end_cen
    df.to_csv(WELL_PLAN_PATH, index=False)

def get_latest_run_folder():
    folders = sorted([d for d in glob.glob(f'{WELL_CSVS_ROOT}/??_??_????__??_??') if os.path.isdir(d)])
    return folders[-1] if folders else None

def parse_summary(folder):
    path = os.path.join(folder, 'simulation_summary.txt')
    if not os.path.exists(path): return None
    text = open(path).read()
    co2 = re.search(r'Total CO2 injected\s*:\s*([\d.]+)', text)
    bhp = re.search(r'Peak injector BHP\s*:\s*([\d.]+)', text)
    inj = re.search(r'Injection end year\s*:\s*([\d.]+)', text)
    if not (co2 and bhp): return None
    return {
        'co2_total_mt': float(co2.group(1)),
        'peak_bhp_bar': float(bhp.group(1)),
        'inj_end_yr':   float(inj.group(1)) if inj else None
    }

def run_matlab():
    folder_before = get_latest_run_folder()
    t0 = time.time()
    cmd = [
        MATLAB_BIN, '-nodesktop', '-nosplash', '-nodisplay', '-r',
        f"addpath('{MRST_ROOT}'); run('{SCRIPT_PATH}'); exit"
    ]
    try:
        subprocess.run(cmd, timeout=MATLAB_TIMEOUT, capture_output=True, text=True)
    except subprocess.TimeoutExpired:
        print('  MATLAB timed out'); return None
    except Exception as e:
        print(f'  MATLAB error: {e}'); return None
    folder_after = get_latest_run_folder()
    if folder_after == folder_before:
        print('  No new run folder detected.'); return None
    print(f'  MATLAB done in {(time.time()-t0)/60:.1f} min -> {os.path.basename(folder_after)}')
    return folder_after

def save_results():
    with open(RESULTS_JSON, 'w') as f:
        json.dump(trial_log, f, indent=2)

print('Utility functions defined.')

Utility functions defined.


In [4]:
# Cell 4: Backup & Verify Setup
backup_well_plan()
assert os.path.exists(MATLAB_BIN), f'MATLAB not found at {MATLAB_BIN}'
print(f'MATLAB binary confirmed.')
assert os.path.exists(SCRIPT_PATH), f'MRST script not found'
print(f'MRST script confirmed.')

df_plan = pd.read_csv(BACKUP_CSV)
active  = df_plan[df_plan['Rate_MtPerYear'] > 0]
print('Currently active wells in well_plan.csv:')
print(active[['Well_Bore_Name','Rate_MtPerYear','Start_Year','End_Year']].to_string(index=False))

latest = get_latest_run_folder()
if latest:
    parsed = parse_summary(latest)
    if parsed:
        print(f'Summary parser test passed on: {os.path.basename(latest)}')
        print(f'  CO2: {parsed["co2_total_mt"]} Mt  |  BHP: {parsed["peak_bhp_bar"]} bar')

print('All checks passed. Ready to run optimization.')

Backup saved -> /Users/apple/Desktop/study/programming/Matlab/Plugins/MRST-2026a/core/examples/data/Johansen/data/well_plan_BACKUP_BO.csv
MATLAB binary confirmed.
MRST script confirmed.
Currently active wells in well_plan.csv:
Well_Bore_Name  Rate_MtPerYear  Start_Year  End_Year
      31/01/01            1.25           0       150
      31/1-3 S            1.25           0       150
        31/2-5            1.25           0       150
        31/4-3            1.25         150       300
      31/05/02            1.25           0       150
      31/05/07            1.25           0       150
      31/07/01            1.25           0        50
Summary parser test passed on: 31_07_2026__03_37
  CO2: 1000.0 Mt  |  BHP: 686.26 bar
All checks passed. Ready to run optimization.


In [5]:
# Cell 5: Define Objective Function
trial_log = []

def objective(trial):
    Q_deep       = trial.suggest_float('Q_deep',       *BOUNDS['Q_deep'],       step=0.05)
    T_start_deep = trial.suggest_int(  'T_start_deep', *BOUNDS['T_start_deep'])
    T_end_deep   = trial.suggest_int(  'T_end_deep',   *BOUNDS['T_end_deep'])
    Q_central    = trial.suggest_float('Q_central',    *BOUNDS['Q_central'],    step=0.05)
    T_start_cen  = trial.suggest_int(  'T_start_cen',  *BOUNDS['T_start_cen'])
    T_end_cen    = trial.suggest_int(  'T_end_cen',    *BOUNDS['T_end_cen'])

    if T_start_deep >= T_end_deep or T_start_cen >= T_end_cen:
        trial_log.append({'trial': trial.number, 'status': 'SKIPPED'})
        return 9999.0

    params = {'Q_deep': Q_deep, 'T_start_deep': T_start_deep, 'T_end_deep': T_end_deep,
              'Q_central': Q_central, 'T_start_cen': T_start_cen, 'T_end_cen': T_end_cen}

    print(f'Trial {trial.number+1}/{N_TRIALS}  Deep: Q={Q_deep:.2f} T=[{T_start_deep}->{T_end_deep}]  '
          f'Central: Q={Q_central:.2f} T=[{T_start_cen}->{T_end_cen}]')

    write_well_plan(Q_deep, T_start_deep, T_end_deep, Q_central, T_start_cen, T_end_cen)
    run_folder = run_matlab()

    if run_folder is None:
        trial_log.append({**params, 'trial': trial.number, 'status': 'FAILED'})
        save_results()
        return 9999.0

    result = parse_summary(run_folder)
    if result is None:
        trial_log.append({**params, 'trial': trial.number, 'status': 'PARSE_ERROR'})
        save_results()
        return 9999.0

    co2_mt  = result['co2_total_mt']
    bhp_bar = result['peak_bhp_bar']
    penalty = PENALTY_LAMBDA * max(0.0, bhp_bar - BHP_LIMIT_BAR)
    reward  = co2_mt - penalty

    print(f'  CO2={co2_mt:.1f} Mt  BHP={bhp_bar:.1f} bar  '
          f'{"[BHP BREACH penalty="+str(round(penalty,1))+"]" if penalty > 0 else "[BHP OK]"}  '
          f'Reward={reward:.1f}')

    entry = {**params, 'trial': trial.number, 'run_folder': os.path.basename(run_folder),
             'status': 'OK', 'co2_mt': co2_mt, 'bhp_bar': bhp_bar,
             'penalty': penalty, 'reward': reward}
    trial_log.append(entry)
    save_results()
    return -reward

print('Objective function defined.')

Objective function defined.


In [6]:
# Cell 6: (Optional) Resume from Previous Run
# Run this before Cell 7 if you are resuming a partially completed optimization.
if os.path.exists(RESULTS_JSON):
    with open(RESULTS_JSON) as f:
        trial_log = json.load(f)
    completed = [t for t in trial_log if t.get('status') == 'OK']
    print(f'Loaded {len(trial_log)} previous trials ({len(completed)} successful).')
    print(f'Trials remaining: {max(0, N_TRIALS - len(trial_log))}')
else:
    trial_log = []
    print('No previous results found. Starting fresh.')

No previous results found. Starting fresh.


In [7]:
# Cell 7: RUN Bayesian Optimization
# Each trial calls MATLAB (~5-10 min). N_TRIALS=40 => ~6 hours overnight.
# Progress auto-saved to bo_results.json after every trial.
# Safe to interrupt with Ctrl+C; re-run Cell 6 + this cell to resume.

print('=' * 60)
print('JOHANSEN CO2 BAYESIAN OPTIMIZATION')
print('=' * 60)
print(f'Trials planned : {N_TRIALS}  |  Warm-up: {N_WARMUP}')
print(f'BHP limit      : {BHP_LIMIT_BAR} bar  |  Penalty lambda: {PENALTY_LAMBDA}')
print('=' * 60)

sampler = TPESampler(n_startup_trials=N_WARMUP, seed=42)
study   = optuna.create_study(study_name='johansen_co2_bo', direction='minimize', sampler=sampler)

n_remaining = max(0, N_TRIALS - len(trial_log))
if n_remaining == 0:
    print('Already have enough trials. Skip to analysis cells.')
else:
    t0 = time.time()
    try:
        study.optimize(objective, n_trials=n_remaining, gc_after_trial=True, show_progress_bar=False)
    except KeyboardInterrupt:
        print('Interrupted. Results saved.')
    elapsed = time.time() - t0
    print(f'Optimization finished in {elapsed/3600:.2f} hours.')
    if study.best_value < 9999:
        print(f'Best reward: {-study.best_value:.2f} Mt CO2')
        print(f'Best params: {study.best_params}')

JOHANSEN CO2 BAYESIAN OPTIMIZATION
Trials planned : 40  |  Warm-up: 10
BHP limit      : 248.0 bar  |  Penalty lambda: 100.0
Trial 1/40  Deep: Q=1.30 T=[19->228]  Central: Q=1.90 T=[3->72]
  MATLAB done in 16.6 min -> 31_07_2026__04_11
Trial 2/40  Deep: Q=0.45 T=[18->192]  Central: Q=2.20 T=[0->292]


[W 2026-07-31 04:30:38,858] Trial 1 failed with parameters: {'Q_deep': 0.45, 'T_start_deep': 18, 'T_end_deep': 192, 'Q_central': 2.2, 'T_start_cen': 0, 'T_end_cen': 292} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/apple/Desktop/study/programming/Python projects/.venv/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_4943/477212746.py", line 23, in objective
    run_folder = run_matlab()
  File "/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_4943/3588100584.py", line 62, in run_matlab
    subprocess.run(cmd, timeout=MATLAB_TIMEOUT, capture_output=True, text=True)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/subprocess.py", line 557, in run
    stdout

Interrupted. Results saved.
Optimization finished in 0.33 hours.


In [ ]:
# Cell 8: Load and Summarize Results
with open(RESULTS_JSON) as f:
    trial_log = json.load(f)

df_results = pd.DataFrame(trial_log)
df_ok = df_results[df_results['status'] == 'OK'].copy()

print(f'Total trials     : {len(df_results)}')
print(f'Successful       : {len(df_ok)}')
print(f'Failed / Skipped : {len(df_results) - len(df_ok)}')

if not df_ok.empty:
    best      = df_ok.loc[df_ok['reward'].idxmax()]
    df_safe   = df_ok[df_ok['bhp_bar'] <= BHP_LIMIT_BAR]
    best_safe = df_safe.loc[df_safe['co2_mt'].idxmax()] if not df_safe.empty else None

    print(f'\nBEST RESULT (max reward):')
    print(f'  Trial #{int(best["trial"])+1}  CO2={best["co2_mt"]:.2f} Mt  BHP={best["bhp_bar"]:.2f} bar  '
          f'{"OK" if best["bhp_bar"] <= BHP_LIMIT_BAR else "BREACH"}')
    print(f'  Deep: Q={best["Q_deep"]:.2f} Mt/yr  T=[{int(best["T_start_deep"])}->{int(best["T_end_deep"])}] yr')
    print(f'  Cen:  Q={best["Q_central"]:.2f} Mt/yr  T=[{int(best["T_start_cen"])}->{int(best["T_end_cen"])}] yr')

    if best_safe is not None:
        print(f'\nBEST SAFE RESULT (BHP <= {BHP_LIMIT_BAR} bar):')
        print(f'  Trial #{int(best_safe["trial"])+1}  CO2={best_safe["co2_mt"]:.2f} Mt  BHP={best_safe["bhp_bar"]:.2f} bar')
        print(f'  Deep: Q={best_safe["Q_deep"]:.2f} Mt/yr  T=[{int(best_safe["T_start_deep"])}->{int(best_safe["T_end_deep"])}] yr')
        print(f'  Cen:  Q={best_safe["Q_central"]:.2f} Mt/yr  T=[{int(best_safe["T_start_cen"])}->{int(best_safe["T_end_cen"])}] yr')
    else:
        print(f'No trial respected BHP <= {BHP_LIMIT_BAR} bar. Lower Q bounds and re-run.')

    print(f'\nCO2 range: {df_ok["co2_mt"].min():.1f} -> {df_ok["co2_mt"].max():.1f} Mt')
    print(f'BHP range: {df_ok["bhp_bar"].min():.1f} -> {df_ok["bhp_bar"].max():.1f} bar')

In [ ]:
# Cell 9: Visualization - Optimization Progress
from matplotlib.lines import Line2D

if df_ok.empty:
    print('No data to plot yet.')
else:
    df_p = df_ok.reset_index(drop=True).copy()
    df_p['trial_num']    = df_p['trial'] + 1
    df_p['cummax_co2']   = df_p['co2_mt'].cummax()
    df_p['cummax_rew']   = df_p['reward'].cummax()
    colors = ['#d62728' if b > BHP_LIMIT_BAR else '#1f77b4' for b in df_p['bhp_bar']]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # Panel 1: CO2 per trial
    ax = axes[0,0]
    ax.scatter(df_p['trial_num'], df_p['co2_mt'], c=colors, zorder=3, s=60)
    ax.plot(df_p['trial_num'], df_p['cummax_co2'], 'k--', lw=1.5, label='Running best')
    ax.legend(handles=[
        Line2D([0],[0],marker='o',color='w',markerfacecolor='#1f77b4',ms=9,label='BHP safe'),
        Line2D([0],[0],marker='o',color='w',markerfacecolor='#d62728',ms=9,label='BHP breach'),
        Line2D([0],[0],color='k',ls='--',label='Running best')
    ], fontsize=9)
    ax.set_xlabel('Trial #'); ax.set_ylabel('CO2 Stored (Mt)')
    ax.set_title('CO2 Stored per Trial', fontweight='bold'); ax.grid(True,ls='--',alpha=0.4)

    # Panel 2: Reward convergence
    ax = axes[0,1]
    ax.plot(df_p['trial_num'], df_p['reward'], 'o-', color='#2ca02c', ms=5, lw=1.5)
    ax.plot(df_p['trial_num'], df_p['cummax_rew'], 'k--', lw=1.5, label='Running best')
    ax.legend(fontsize=9); ax.set_xlabel('Trial #'); ax.set_ylabel('Reward (CO2 - Penalty)')
    ax.set_title('Reward Convergence (BO Learning Curve)', fontweight='bold'); ax.grid(True,ls='--',alpha=0.4)

    # Panel 3: BHP vs CO2
    ax = axes[1,0]
    sc = ax.scatter(df_p['bhp_bar'], df_p['co2_mt'], c=df_p['trial_num'], cmap='plasma', s=60, zorder=3)
    ax.axvline(BHP_LIMIT_BAR, color='red', ls='--', lw=2, label=f'BHP limit ({BHP_LIMIT_BAR} bar)')
    plt.colorbar(sc, ax=ax, label='Trial #')
    ax.legend(fontsize=9); ax.set_xlabel('Peak BHP (bar)'); ax.set_ylabel('CO2 Stored (Mt)')
    ax.set_title('Peak BHP vs CO2 Stored', fontweight='bold'); ax.grid(True,ls='--',alpha=0.4)

    # Panel 4: Q landscape
    ax = axes[1,1]
    sc2 = ax.scatter(df_p['Q_deep'], df_p['Q_central'], c=df_p['co2_mt'],
                     cmap='YlGn', s=70, zorder=3, edgecolors='k', lw=0.5)
    best_row = df_p.loc[df_p['reward'].idxmax()]
    ax.scatter([best_row['Q_deep']], [best_row['Q_central']], s=200, marker='*', color='red', zorder=5, label='Best')
    plt.colorbar(sc2, ax=ax, label='CO2 Stored (Mt)')
    ax.legend(fontsize=9); ax.set_xlabel('Q_deep (Mt/yr)'); ax.set_ylabel('Q_central (Mt/yr)')
    ax.set_title('Injection Rate Landscape', fontweight='bold'); ax.grid(True,ls='--',alpha=0.4)

    plt.suptitle('Bayesian Optimization - Johansen CO2 Storage', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    out_path = f'{MRST_ROOT}/core/examples/data/Johansen/python/bo_progress.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Progress plot saved to {out_path}')

In [ ]:
# Cell 10: Parameter Importance (Random Forest)
from sklearn.ensemble import RandomForestRegressor

if df_ok.empty or len(df_ok) < 5:
    print('Need at least 5 successful trials for importance analysis.')
else:
    param_cols = ['Q_deep','T_start_deep','T_end_deep','Q_central','T_start_cen','T_end_cen']
    X = df_ok[param_cols].values
    y = df_ok['reward'].values
    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(X, y)
    imp = rf.feature_importances_
    sidx = np.argsort(imp)[::-1]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar([param_cols[i] for i in sidx], imp[sidx],
                  color=cm.plasma(np.linspace(0.15, 0.85, len(param_cols))), edgecolor='k', lw=0.7)
    for bar, v in zip(bars, imp[sidx]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel('Feature Importance (RF)'); ax.set_title('Parameter Importance', fontweight='bold')
    ax.grid(True, ls='--', alpha=0.4, axis='y')
    plt.tight_layout()
    plt.savefig(f'{MRST_ROOT}/core/examples/data/Johansen/python/bo_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Parameter importance (most to least):')
    for i in sidx:
        print(f'  {param_cols[i]:20s}: {imp[i]:.1%}')

In [ ]:
# Cell 11: Surrogate Surface Visualization
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

if df_ok.empty or len(df_ok) < 5:
    print('Need at least 5 successful trials.')
else:
    param_cols = ['Q_deep','T_start_deep','T_end_deep','Q_central','T_start_cen','T_end_cen']
    X = df_ok[param_cols].values
    y = df_ok['co2_mt'].values
    scaler = StandardScaler()
    surr   = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
    surr.fit(scaler.fit_transform(X), y)

    best_params = df_ok.loc[df_ok['reward'].idxmax(), param_cols].values

    Q_grid = np.linspace(*BOUNDS['Q_deep'], 40)
    T_grid = np.linspace(*BOUNDS['T_end_deep'], 40)
    QQ, TT = np.meshgrid(Q_grid, T_grid)

    grid_pts = np.column_stack([QQ.ravel(), np.full(QQ.size, best_params[1]),
                                TT.ravel(), np.full(QQ.size, best_params[3]),
                                np.full(QQ.size, best_params[4]), np.full(QQ.size, best_params[5])])
    Z = surr.predict(scaler.transform(grid_pts)).reshape(QQ.shape)

    fig, ax = plt.subplots(figsize=(9, 6))
    cf = ax.contourf(QQ, TT, Z, levels=25, cmap='YlOrRd')
    ax.contour(QQ, TT, Z, levels=10, colors='k', linewidths=0.5, alpha=0.5)
    plt.colorbar(cf, ax=ax, label='Predicted CO2 Stored (Mt)')
    ax.scatter(df_ok['Q_deep'], df_ok['T_end_deep'], c=df_ok['co2_mt'],
               cmap='YlOrRd', edgecolors='k', s=60, zorder=5, lw=0.8)
    ax.scatter([best_params[0]], [best_params[2]], s=250, marker='*', color='blue', zorder=6, label='Best')
    ax.set_xlabel('Q_deep (Mt/yr)'); ax.set_ylabel('T_end_deep (yr)')
    ax.set_title('Surrogate Surface: CO2 Stored vs Q_deep & T_end_deep\n(other params held at best values)',
                 fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{MRST_ROOT}/core/examples/data/Johansen/python/bo_surrogate.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Surrogate surface saved.')

In [ ]:
# Cell 12: Full Results Table (all trials, sorted by reward)
from IPython.display import display

if df_ok.empty:
    print('No results yet.')
else:
    dcols = ['trial','Q_deep','T_start_deep','T_end_deep','Q_central','T_start_cen','T_end_cen',
             'co2_mt','bhp_bar','penalty','reward']
    df_show = df_ok[dcols].copy()
    df_show['trial'] = df_show['trial'].astype(int) + 1
    df_show = df_show.sort_values('reward', ascending=False).reset_index(drop=True)

    def style_row(row):
        out = []
        for col in row.index:
            if col == 'bhp_bar' and row['bhp_bar'] > BHP_LIMIT_BAR:
                out.append('background-color:#ffcccc')
            elif col == 'reward' and row.name == 0:
                out.append('background-color:#ccffcc;font-weight:bold')
            else:
                out.append('')
        return out

    styled = df_show.style.apply(style_row, axis=1).format(
        {'Q_deep':'{:.2f}','Q_central':'{:.2f}','co2_mt':'{:.2f}',
         'bhp_bar':'{:.2f}','penalty':'{:.1f}','reward':'{:.2f}'}
    )
    print('All trials sorted by reward (best first). Green=best, Red=BHP breach.')
    display(styled)

In [ ]:
# Cell 13: Apply Optimal Parameters to well_plan.csv
# APPLY_MODE options:
#   'best_safe'   -> best CO2 with BHP <= limit (recommended)
#   'best_reward' -> best raw reward (may include BHP breaches)
APPLY_MODE = 'best_safe'

if df_ok.empty:
    print('No results to apply.')
else:
    df_safe = df_ok[df_ok['bhp_bar'] <= BHP_LIMIT_BAR]
    if APPLY_MODE == 'best_safe' and df_safe.empty:
        print(f'No safe trials (BHP <= {BHP_LIMIT_BAR} bar). Applying best_reward instead.')
        APPLY_MODE = 'best_reward'
    if APPLY_MODE == 'best_safe':
        chosen   = df_safe.loc[df_safe['co2_mt'].idxmax()]
        mode_str = 'Best Safe (BHP-constrained)'
    else:
        chosen   = df_ok.loc[df_ok['reward'].idxmax()]
        mode_str = 'Best Reward (unconstrained)'

    print(f'Applying [{mode_str}] parameters to well_plan.csv...')
    print(f'  Deep: Q={chosen["Q_deep"]:.2f} Mt/yr  T=[{int(chosen["T_start_deep"])}->{int(chosen["T_end_deep"])}] yr')
    print(f'  Cen:  Q={chosen["Q_central"]:.2f} Mt/yr  T=[{int(chosen["T_start_cen"])}->{int(chosen["T_end_cen"])}] yr')
    print(f'  Expected CO2: {chosen["co2_mt"]:.2f} Mt  |  BHP: {chosen["bhp_bar"]:.2f} bar')

    write_well_plan(chosen['Q_deep'],    int(chosen['T_start_deep']), int(chosen['T_end_deep']),
                    chosen['Q_central'], int(chosen['T_start_cen']),  int(chosen['T_end_cen']))

    print('well_plan.csv updated with optimal parameters!')
    print(f'Run example3DJohansen.m in MATLAB to validate the final simulation.')
    print(f'Backup at: {BACKUP_CSV}')

    df_updated = pd.read_csv(WELL_PLAN_PATH)
    active_upd = df_updated[df_updated['Rate_MtPerYear'] > 0]
    print('Updated well_plan.csv (active wells):')
    print(active_upd[['Well_Bore_Name','Rate_MtPerYear','Start_Year','End_Year']].to_string(index=False))

In [ ]:
# Cell 14: Emergency Restore
# Run ONLY if you want to undo ALL changes and restore the original well_plan.csv.

# Uncomment the line below to restore:
# restore_well_plan()

print('To restore original well_plan.csv, uncomment restore_well_plan() above.')